In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

def run_churn_ml_pipeline(input_csv_path, output_csv_path):
    print("1. Loading dataset...")
    df = pd.read_csv(input_csv_path)

    # Check if critical identifiers exist
    has_customer_id = 'CustomerID' in df.columns

    # Prepare features (X) and target (y)
    X_raw = df.drop(columns=['CustomerID', 'Churn'] if has_customer_id else ['Churn'])
    y = df['Churn']

    print("2. Preprocessing & One-Hot Encoding categorical features...")
    categorical_cols = ['Gender', 'Subscription Type', 'Contract Length']
    X_encoded = pd.get_dummies(X_raw, columns=categorical_cols, drop_first=True)

    print("3. Splitting into Train & Test sets...")
    X_train, X_test, y_train, y_test = train_test_split(
        X_encoded, y, test_size=0.2, random_state=42, stratify=y
    )

    print("4. Scaling numerical features...")
    scaler = StandardScaler()
    num_cols = ['Age', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Total Spend', 'Last Interaction']

    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()

    X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

    print("5. Training Random Forest Classifier...")
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Predictions & Evaluation
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    print("\n--- Model Performance Evaluation ---")
    print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
    print(f"ROC AUC Score:  {roc_auc_score(y_test, y_proba):.4f}")
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

    print("6. Generating Churn Probabilities & Risk Scores for Full Dataset...")
    # Scale full feature set for export prediction
    X_full_scaled = X_encoded.copy()
    X_full_scaled[num_cols] = scaler.transform(X_encoded[num_cols])

    full_probabilities = model.predict_proba(X_full_scaled)[:, 1]

    # Append Machine Learning columns directly
    df['Churn_Probability'] = np.round(full_probabilities, 4)
    df['Predicted_Churn_Risk'] = np.where(df['Churn_Probability'] >= 0.5, 'High Risk', 'Low Risk')

    print(f"7. Saving updated ML dataset to {output_csv_path}...")
    df.to_csv(output_csv_path, index=False)
    print("Process completed successfully!")

if __name__ == "__main__":
    # Specify input file name and output file name
    INPUT_FILE = "customer_churn_cleaned clea data.csv"
    OUTPUT_FILE = "customer_churn_with_ml.csv"

    run_churn_ml_pipeline(INPUT_FILE, OUTPUT_FILE)

1. Loading dataset...
2. Preprocessing & One-Hot Encoding categorical features...
3. Splitting into Train & Test sets...
4. Scaling numerical features...
5. Training Random Forest Classifier...

--- Model Performance Evaluation ---
Accuracy Score: 0.9981
ROC AUC Score:  1.0000

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      6776
           1       1.00      1.00      1.00      6099

    accuracy                           1.00     12875
   macro avg       1.00      1.00      1.00     12875
weighted avg       1.00      1.00      1.00     12875

6. Generating Churn Probabilities & Risk Scores for Full Dataset...
7. Saving updated ML dataset to customer_churn_with_ml.csv...
Process completed successfully!
